# Guía PC2 · De tu base de datos a un modelo de machine learning

**Google Colab · ejecución secuencial · decisiones explícitas · variables nuevas en cada etapa**

Esta guía sirve para datos tabulares supervisados: clasificación, regresión y una serie de tiempo.
Tú defines el problema, el target, las columnas, la limpieza, los hiperparámetros y el modelo final.
El código calcula y muestra resultados; no elige esas decisiones por ti.

**Cómo recorrerla**
1. Ejecuta instalación, importaciones y carga. Configura tus columnas después de mirar la vista inicial.
2. Reserva validación y prueba. Haz el EDA de entrenamiento y decide las limpiezas opcionales.
3. Ejecuta la preparación y el recorrido elegido. Los otros recorridos se omiten por `TIPO_PROBLEMA`.
4. Compara validación, escribe tu modelo elegido y ejecuta la evaluación final una vez.
5. Carga una base nueva y exporta sus predicciones. El target no es necesario para predecir.

**Historial visible:** `df_original` → `df_reglas` → `df_reserva_train` → `df_train_limpio` →
`X_train`. Cada transformación tiene otro nombre; no se usan `df = df.drop(...)` ni `inplace=True`.
Los nombres de iteración se reutilizan dentro de los bucles, pero sus entradas no se modifican.

No existe una preparación universal para texto libre, imágenes, paneles de varias series o cualquier
estructura imaginable. Aquí debes declarar predictores tabulares y, en series, una fecha por periodo.
Si cambias una configuración, reinicia el entorno y ejecuta desde el principio para evitar estado antiguo.

## 0. Entorno de Google Colab
La siguiente celda instala las dependencias en el entorno de ejecución. Si Colab pide reiniciar,
reinícialo y continúa desde las importaciones. SHAP se importa solo si activas su sección.
Las versiones efectivamente usadas se guardan junto al modelo.

In [ ]:
%pip install -q "pandas>=2.2,<4" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "xgboost>=2.1,<4" "lightgbm>=4.5,<5" "shap>=0.46,<1" openpyxl joblib

In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import copy
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, mean_absolute_error, mean_squared_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
versiones_entorno = {p: version(p) for p in ["pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "shap", "joblib"]}
display(pd.Series(versiones_entorno, name="Versión"))

## 1. Carga tu archivo
En Colab puedes subirlo desde el panel **Archivos**, montarlo desde Drive o activar `SUBIR_ARCHIVO`.
Indica un solo archivo por carga; no se selecciona silenciosamente uno entre varios.
`FORMATO_ARCHIVO` acepta `csv` o `excel`. Ajusta separador y decimales a tu archivo.

In [ ]:
SUBIR_ARCHIVO = False
RUTA_ARCHIVO = "/content/mi_base.csv"
FORMATO_ARCHIVO = "csv"  # "csv" o "excel"
SEPARADOR = ","
CODIFICACION = "utf-8"
DECIMAL = "."
HOJA_EXCEL = 0

In [ ]:
if SUBIR_ARCHIVO:
    from google.colab import files
    archivos_subidos = files.upload()
    if len(archivos_subidos) != 1:
        raise ValueError("Sube exactamente un archivo y vuelve a ejecutar esta celda.")
    ruta_lectura = Path(next(iter(archivos_subidos)))
else:
    ruta_lectura = Path(RUTA_ARCHIVO)

if not ruta_lectura.is_file():
    raise FileNotFoundError(f"No existe {ruta_lectura}. Sube tu base y ajusta RUTA_ARCHIVO.")
if FORMATO_ARCHIVO == "csv":
    df_original = pd.read_csv(ruta_lectura, sep=SEPARADOR, encoding=CODIFICACION, decimal=DECIMAL)
elif FORMATO_ARCHIVO == "excel":
    df_original = pd.read_excel(ruta_lectura, sheet_name=HOJA_EXCEL, decimal=DECIMAL)
else:
    raise ValueError("FORMATO_ARCHIVO debe ser csv o excel.")

if df_original.empty or not df_original.columns.is_unique:
    raise ValueError("La base está vacía o tiene nombres de columnas duplicados.")
print("Filas, columnas:", df_original.shape)
display(df_original.head())
display(df_original.dtypes.to_frame("Tipo leído"))
print("Candidatas numéricas:", df_original.select_dtypes(include="number").columns.tolist())
print("Candidatas categóricas:", df_original.select_dtypes(exclude="number").columns.tolist())

## 2. Configuración del problema: edita esta celda

| Parámetro | Qué decides |
|---|---|
| `TIPO_PROBLEMA` | `clasificacion`, `regresion` o `serie_tiempo` |
| `TARGET` | Columna que quieres predecir; nunca debe estar entre los predictores |
| `COLUMNAS_NUMERICAS` / `COLUMNAS_CATEGORICAS` | Listas confirmadas por ti; incluye como categóricas los códigos que no expresan cantidades |
| `COLUMNAS_ID` / `COLUMNAS_EXCLUIR` | Columnas conservadas para trazabilidad o descartadas del modelo |
| `FRACCION_VALIDACION` / `FRACCION_PRUEBA` | Proporciones del total, no del remanente |
| `CLASE_POSITIVA` | Etiqueta original del evento de interés; solo clasificación binaria |
| `IMPUTAR_*`, `ESCALAR`, `CATEGORIA_DESCONOCIDA` | Política explícita de preparación |

Para clasificación usa observaciones independientes; si una persona aparece en varias filas,
necesitas diseñar una partición por grupos antes de usar este recorrido.
Para series deja vacías las listas de predictores: esta versión construye calendario y rezagos del target.
La fecha se declara por separado. No se usan variables externas futuras en esta versión.

In [ ]:
TIPO_PROBLEMA = "clasificacion"
TARGET = "CAMBIA_POR_TU_TARGET"
COLUMNAS_NUMERICAS = []  # Ejemplo: ["edad", "saldo"]
COLUMNAS_CATEGORICAS = []  # Ejemplo: ["pais", "canal"]
COLUMNAS_ID = []  # Ejemplo: ["cliente_id"]
COLUMNAS_EXCLUIR = []  # Ejemplo: ["resultado_posterior"] (fuga de información)
SEMILLA = 42
FRACCION_VALIDACION = 0.20
FRACCION_PRUEBA = 0.20
CLASE_POSITIVA = 1  # Puede ser "Sí", "Abandona", etc.
UMBRAL = 0.50

IMPUTAR_NUMERICAS = False
ESTRATEGIA_NUMERICA = "median"  # "mean", "median", "most_frequent" o "constant"
VALOR_NUMERICO_CONSTANTE = 0
IMPUTAR_CATEGORICAS = False
ESTRATEGIA_CATEGORICA = "most_frequent"  # O "constant"
VALOR_CATEGORICO_CONSTANTE = "Sin dato"
ESCALAR = True
CATEGORIA_DESCONOCIDA = "ignore"  # "ignore": todo cero; "error": detenerse

FECHA = "fecha"
FORMATO_FECHA_SERIE = "%Y-%m-%d"
FRECUENCIA = "D"  # "D": diaria, "W-MON": semanal, "MS": inicio de mes
REZAGOS = [1, 7, 14]  # En periodos de FRECUENCIA
VENTANAS = [7]
HORIZONTE = 14
VENTANAS_BACKTEST = 3
PASO_BACKTEST = 7
AGREGACION_SERIE = None  # None, "sum" o "mean": tú decides si agregas periodos
PERMITIR_ORDENAR_FECHAS = False

### 2.1 Reglas deterministas que también aplicarás a datos nuevos
No se aprenden medianas ni categorías aquí. Por defecto todas las reglas están vacías.
Si conviertes un texto a número, utiliza una expresión adecuada a tu archivo antes de convertirlo;
no se adivinan separadores de miles ni símbolos monetarios. Las conversiones inválidas detienen la ejecución.

**`aplicar_reglas(datos, numericas, textos, fechas, reemplazos)`**
- `datos`: DataFrame de entrada, que no se modifica.
- `numericas`: nombres que deseas convertir a número.
- `textos`: nombres a convertir a texto, quitar espacios extremos y pasar a minúsculas.
- `fechas`: diccionario `{columna: formato}` para convertir fechas.
- `reemplazos`: diccionario `{columna: {valor_original: valor_nuevo}}`.
- Retorna un DataFrame nuevo después de cuatro etapas con nombres distintos.

Ejemplo: `aplicar_reglas(pd.DataFrame({"edad": ["20"], "pais": [" Perú "]}), ["edad"], ["pais"], {}, {})`.
Salida esperada: una fila con `edad=20` y `pais="perú"`; la entrada permanece igual.
No normalices el target accidentalmente: si lo haces, actualiza `CLASE_POSITIVA` a la etiqueta resultante.

In [ ]:
def aplicar_reglas(datos, numericas, textos, fechas, reemplazos):
    """Aplica solo reglas declaradas; devuelve otra tabla sin modificar datos."""
    columnas_reglas = set(numericas) | set(textos) | set(fechas) | set(reemplazos)
    columnas_ausentes = columnas_reglas - set(datos.columns)
    if columnas_ausentes:
        raise ValueError(f"Faltan columnas para las reglas: {sorted(columnas_ausentes)}")
    etapa_reemplazos = datos.replace(reemplazos)
    etapa_numerica = etapa_reemplazos.assign(**{
        c: pd.to_numeric(etapa_reemplazos[c], errors="raise") for c in numericas
    })
    etapa_texto = etapa_numerica.assign(**{
        c: etapa_numerica[c].astype("string").str.strip().str.lower().astype(object)
            .where(etapa_numerica[c].notna(), np.nan) for c in textos
    })
    etapa_fechas = etapa_texto.assign(**{
        c: pd.to_datetime(etapa_texto[c], format=f, errors="raise") for c, f in fechas.items()
    })
    return etapa_fechas

REGLAS_NUMERICAS = []
REGLAS_TEXTOS = []
REGLAS_FECHAS = {}  # Ejemplo: {"fecha_alta": "%d/%m/%Y"}
REGLAS_REEMPLAZOS = {}  # Ejemplo: {"pais": {"PE": "Perú"}}
df_reglas = aplicar_reglas(df_original, REGLAS_NUMERICAS, REGLAS_TEXTOS, REGLAS_FECHAS, REGLAS_REEMPLAZOS)
display(df_reglas.head())

### 2.2 Comprobar la configuración antes de reservar datos
Los nulos del target se reportan como error: decide qué hacer con esas filas en la base de origen.
No se imputa el objetivo. Las columnas declaradas como ID o excluidas no pueden ser predictoras.

In [ ]:
if TIPO_PROBLEMA not in {"clasificacion", "regresion", "serie_tiempo"}:
    raise ValueError("TIPO_PROBLEMA no reconocido.")
if TARGET not in df_reglas:
    raise ValueError(f"TARGET={TARGET!r} no existe. Edita la configuración.")
if df_reglas[TARGET].isna().any():
    raise ValueError(f"Hay {df_reglas[TARGET].isna().sum()} targets nulos. Decide cómo resolverlos.")
if not (0 < FRACCION_VALIDACION < 1 and 0 < FRACCION_PRUEBA < 1 and FRACCION_VALIDACION + FRACCION_PRUEBA < 1):
    raise ValueError("Las fracciones deben ser positivas y sumar menos de 1.")
if not 0 <= UMBRAL <= 1:
    raise ValueError("UMBRAL debe estar entre 0 y 1.")
if CATEGORIA_DESCONOCIDA not in {"ignore", "error"}:
    raise ValueError("CATEGORIA_DESCONOCIDA debe ser ignore o error.")
predictores_declarados = COLUMNAS_NUMERICAS + COLUMNAS_CATEGORICAS
columnas_declaradas = predictores_declarados + COLUMNAS_ID + COLUMNAS_EXCLUIR
if set(columnas_declaradas) - set(df_reglas.columns):
    raise ValueError(f"Columnas inexistentes: {set(columnas_declaradas) - set(df_reglas.columns)}")
if len(predictores_declarados) != len(set(predictores_declarados)):
    raise ValueError("Un predictor está repetido o declarado como numérico y categórico.")
if set(predictores_declarados) & ({TARGET} | set(COLUMNAS_ID) | set(COLUMNAS_EXCLUIR)):
    raise ValueError("El target, ID y exclusiones no pueden estar entre los predictores.")
if TIPO_PROBLEMA != "serie_tiempo" and not predictores_declarados:
    raise ValueError("Confirma al menos un predictor en las listas de columnas.")
if TIPO_PROBLEMA == "serie_tiempo" and predictores_declarados:
    raise ValueError("Este recorrido temporal usa calendario y rezagos. Deja vacías las listas de predictores.")
if TIPO_PROBLEMA != "clasificacion":
    if not pd.api.types.is_numeric_dtype(df_reglas[TARGET]) or not np.isfinite(df_reglas[TARGET].to_numpy(dtype=float)).all():
        raise ValueError("El target de regresión/serie debe ser numérico y finito. Configura las conversiones.")
if len(df_reglas) < 15:
    raise ValueError("Se requieren al menos 15 filas; según clases y rezagos puedes necesitar muchas más.")

## 3. Reservar entrenamiento, validación y prueba
**No inspecciones prueba para tomar decisiones.** La revisión inicial solo confirmó estructura y target.
Desde aquí, el EDA y las decisiones de limpieza usan `df_reserva_train`.
En clasificación se estratifica; si faltan ejemplos de alguna clase, se detiene en lugar de cambiar el método.
En regresión se divide aleatoriamente. En series se reservan los últimos periodos cronológicamente.

En series, ordenar o agregar exige tu autorización mediante los parámetros. No se rellenan huecos.
La agregación define la unidad temporal, no aprende parámetros del target; las ventanas se separan después.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    if TIPO_PROBLEMA == "clasificacion":
        conteo_clases_inicial = df_reglas[TARGET].value_counts()
        if len(conteo_clases_inicial) < 2 or conteo_clases_inicial.min() < 5:
            raise ValueError("Clasificación requiere al menos dos clases y cinco filas por clase para este esquema.")
    try:
        df_desarrollo, df_reserva_prueba = train_test_split(
            df_reglas, test_size=FRACCION_PRUEBA, random_state=SEMILLA,
            stratify=df_reglas[TARGET] if TIPO_PROBLEMA == "clasificacion" else None)
        df_reserva_train, df_reserva_validacion = train_test_split(
            df_desarrollo, test_size=FRACCION_VALIDACION / (1 - FRACCION_PRUEBA), random_state=SEMILLA,
            stratify=df_desarrollo[TARGET] if TIPO_PROBLEMA == "clasificacion" else None)
    except ValueError as error_particion:
        raise ValueError("No se pudo dividir. Revisa tamaño, clases y fracciones sin desactivar silenciosamente la estratificación.") from error_particion

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    if FECHA not in df_reglas or FECHA == TARGET:
        raise ValueError("Declara una columna FECHA existente y distinta del target.")
    if not REZAGOS or any(not isinstance(k, int) or k < 1 for k in REZAGOS + VENTANAS):
        raise ValueError("REZAGOS no puede estar vacío y los rezagos/ventanas deben ser enteros positivos.")
    if HORIZONTE < 1 or VENTANAS_BACKTEST < 1 or PASO_BACKTEST < 1:
        raise ValueError("Horizonte, ventanas de backtesting y paso deben ser positivos.")
    df_serie_fechas = df_reglas.assign(**{FECHA: pd.to_datetime(df_reglas[FECHA], format=FORMATO_FECHA_SERIE, errors="raise")})
    if df_serie_fechas[FECHA].isna().any():
        raise ValueError("Hay fechas vacías.")
    print("Fechas repetidas:", df_serie_fechas[FECHA].duplicated().sum())
    print("Orden cronológico:", df_serie_fechas[FECHA].is_monotonic_increasing)
    if not df_serie_fechas[FECHA].is_monotonic_increasing and not PERMITIR_ORDENAR_FECHAS:
        raise ValueError("Fechas desordenadas. Revisa y activa PERMITIR_ORDENAR_FECHAS si corresponde.")
    df_serie_ordenada = df_serie_fechas.sort_values(FECHA) if PERMITIR_ORDENAR_FECHAS else df_serie_fechas.copy()
    if AGREGACION_SERIE is None:
        if df_serie_ordenada[FECHA].duplicated().any():
            raise ValueError("Hay varias filas por fecha. Decide una agregación explícita.")
        serie_periodos = df_serie_ordenada.set_index(FECHA)[TARGET].copy()
    elif AGREGACION_SERIE in {"sum", "mean"}:
        agrupacion_periodos = df_serie_ordenada.set_index(FECHA)[TARGET].resample(FRECUENCIA)
        serie_periodos = agrupacion_periodos.sum(min_count=1) if AGREGACION_SERIE == "sum" else agrupacion_periodos.mean()
    else:
        raise ValueError("AGREGACION_SERIE debe ser None, sum o mean.")
    fechas_esperadas = pd.date_range(serie_periodos.index.min(), serie_periodos.index.max(), freq=FRECUENCIA)
    fechas_faltantes = fechas_esperadas.difference(serie_periodos.index)
    fechas_fuera_frecuencia = serie_periodos.index.difference(fechas_esperadas)
    print("Periodos faltantes:", fechas_faltantes.tolist()[:20])
    print("Fechas fuera de frecuencia:", fechas_fuera_frecuencia.tolist()[:20])
    if len(fechas_faltantes) or len(fechas_fuera_frecuencia) or serie_periodos.isna().any():
        raise ValueError("La serie debe tener un valor por periodo sin huecos. Decide cómo corregirla; no se rellenará automáticamente.")
    df_serie_regular = serie_periodos.rename(TARGET).rename_axis(FECHA).reset_index()
    n_prueba_temporal = max(1, int(np.ceil(len(df_serie_regular) * FRACCION_PRUEBA)))
    n_validacion_temporal = max(1, int(np.ceil(len(df_serie_regular) * FRACCION_VALIDACION)))
    corte_train_temporal = len(df_serie_regular) - n_prueba_temporal - n_validacion_temporal
    corte_validacion_temporal = len(df_serie_regular) - n_prueba_temporal
    memoria_temporal = max(REZAGOS + VENTANAS)
    if corte_train_temporal <= memoria_temporal + 5:
        raise ValueError("Historia insuficiente para los rezagos y las particiones elegidas.")
    df_reserva_train = df_serie_regular.iloc[:corte_train_temporal].copy()
    df_reserva_validacion = df_serie_regular.iloc[corte_train_temporal:corte_validacion_temporal].copy()
    df_reserva_prueba = df_serie_regular.iloc[corte_validacion_temporal:].copy()

display(pd.DataFrame({"Filas": [len(df_reserva_train), len(df_reserva_validacion), len(df_reserva_prueba)]},
                     index=["Entrenamiento", "Validación", "Prueba reservada"]))

## 4. EDA: conocer entrenamiento antes de limpiar
`COLUMNAS_EDA_NUMERICAS`, `COLUMNAS_EDA_CATEGORICAS` y `MAX_GRAFICOS` controlan la exploración.
Los tipos detectados solo orientan el diagnóstico; no cambian tus listas de predictores.
Correlación no implica causalidad. Una categoría con muchas etiquetas puede ser un ID o texto libre.

In [ ]:
COLUMNAS_EDA_NUMERICAS = COLUMNAS_NUMERICAS if TIPO_PROBLEMA != "serie_tiempo" else [TARGET]
COLUMNAS_EDA_CATEGORICAS = COLUMNAS_CATEGORICAS
MAX_GRAFICOS = 6
TOP_CATEGORIAS = 15
BINS = 30
FACTOR_IQR = 1.5

perfil_calidad = pd.DataFrame({
    "tipo": df_reserva_train.dtypes.astype(str),
    "nulos": df_reserva_train.isna().sum(),
    "nulos_pct": df_reserva_train.isna().mean() * 100,
    "valores_distintos": df_reserva_train.nunique(dropna=True),
    "constante": df_reserva_train.nunique(dropna=True).eq(1),
    "infinito": {c: np.isinf(df_reserva_train[c].to_numpy(dtype=float)).sum()
        if pd.api.types.is_numeric_dtype(df_reserva_train[c]) else 0 for c in df_reserva_train.columns}
})
display(perfil_calidad)
display(df_reserva_train.describe(include="all").T)
print("Filas duplicadas completas:", df_reserva_train.duplicated().sum())
if COLUMNAS_ID and set(COLUMNAS_ID).issubset(df_reserva_train.columns):
    print("IDs repetidos:", df_reserva_train.duplicated(subset=COLUMNAS_ID).sum())
plt.figure(figsize=(10, 3))
sns.heatmap(df_reserva_train.head(200).isna(), cbar=False)
plt.title("Nulos: primeras 200 filas de entrenamiento")
plt.show()

In [ ]:
for columna_eda in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
    if not pd.api.types.is_numeric_dtype(df_reserva_train[columna_eda]):
        raise TypeError(f"{columna_eda} no es numérica; configura una conversión explícita.")
    fig_distribucion, ejes_distribucion = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df_reserva_train[columna_eda], bins=BINS, ax=ejes_distribucion[0])
    sns.boxplot(x=df_reserva_train[columna_eda], ax=ejes_distribucion[1])
    fig_distribucion.suptitle(columna_eda)
    plt.tight_layout()
    plt.show()

for columna_frecuencia in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
    tabla_frecuencias = pd.concat([
        df_reserva_train[columna_frecuencia].value_counts(dropna=False).rename("Frecuencia"),
        df_reserva_train[columna_frecuencia].value_counts(dropna=False, normalize=True).mul(100).rename("Porcentaje")
    ], axis=1)
    display(tabla_frecuencias.head(TOP_CATEGORIAS))
    tabla_frecuencias["Frecuencia"].head(TOP_CATEGORIAS).plot.bar(title=columna_frecuencia, figsize=(8, 3))
    plt.show()

if TIPO_PROBLEMA == "clasificacion":
    display(df_reserva_train[TARGET].value_counts(dropna=False).to_frame("Distribución del target"))
    df_reserva_train[TARGET].value_counts().plot.bar(title="Clases en entrenamiento")
    plt.show()
else:
    sns.histplot(df_reserva_train[TARGET], bins=BINS)
    plt.title("Distribución del target")
    plt.show()

### 4.1 Atípicos: diagnóstico, no eliminación
Los límites IQR se calculan sobre entrenamiento. Un valor fuera de esos límites puede ser correcto.
`indices_atipicos` solo lista candidatos para inspeccionar; no se filtra ninguna observación.

In [ ]:
cuartil_inferior = df_reserva_train[COLUMNAS_EDA_NUMERICAS].quantile(0.25)
cuartil_superior = df_reserva_train[COLUMNAS_EDA_NUMERICAS].quantile(0.75)
rango_intercuartil = cuartil_superior - cuartil_inferior
limites_iqr = pd.DataFrame({"inferior": cuartil_inferior - FACTOR_IQR * rango_intercuartil,
                             "superior": cuartil_superior + FACTOR_IQR * rango_intercuartil})
mascara_atipicos = df_reserva_train[COLUMNAS_EDA_NUMERICAS].lt(limites_iqr["inferior"]) | df_reserva_train[COLUMNAS_EDA_NUMERICAS].gt(limites_iqr["superior"])
indices_atipicos = df_reserva_train.index[mascara_atipicos.any(axis=1)].tolist()
display(limites_iqr.assign(casos_fuera=mascara_atipicos.sum()))
display(df_reserva_train.loc[indices_atipicos].head(20))
print("Candidatos a revisar, sin eliminar:", len(indices_atipicos))

In [ ]:
columnas_correlacion = list(dict.fromkeys(COLUMNAS_EDA_NUMERICAS + ([TARGET] if TIPO_PROBLEMA != "clasificacion" else [])))
if len(columnas_correlacion) >= 2:
    matriz_correlacion = df_reserva_train[columnas_correlacion].corr()
    plt.figure(figsize=(9, 7))
    sns.heatmap(matriz_correlacion, cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title("Correlación entre variables numéricas · entrenamiento")
    plt.show()

if TIPO_PROBLEMA == "clasificacion":
    for columna_bivariada in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
        sns.boxplot(data=df_reserva_train, x=TARGET, y=columna_bivariada)
        plt.title(f"{columna_bivariada} por clase")
        plt.show()
    for columna_cruce in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
        tabla_cruzada = pd.crosstab(df_reserva_train[columna_cruce], df_reserva_train[TARGET], normalize="index")
        display(tabla_cruzada.head(TOP_CATEGORIAS))
elif TIPO_PROBLEMA == "regresion":
    for columna_dispersion in COLUMNAS_EDA_NUMERICAS[:MAX_GRAFICOS]:
        sns.scatterplot(data=df_reserva_train, x=columna_dispersion, y=TARGET, alpha=0.5)
        plt.show()
    for columna_segmento_eda in COLUMNAS_EDA_CATEGORICAS[:MAX_GRAFICOS]:
        display(df_reserva_train.groupby(columna_segmento_eda, dropna=False)[TARGET].agg(["count", "mean", "median"]).head(TOP_CATEGORIAS))
else:
    df_reserva_train.plot(x=FECHA, y=TARGET, figsize=(12, 4), title="Serie de entrenamiento")
    plt.show()
    resumen_calendario = df_reserva_train.assign(dia_semana=df_reserva_train[FECHA].dt.dayofweek,
        mes=df_reserva_train[FECHA].dt.month)
    display(resumen_calendario.groupby("dia_semana")[TARGET].agg(["count", "mean", "median"]))
    display(resumen_calendario.groupby("mes")[TARGET].agg(["count", "mean", "median"]))

## 5. Limpieza opcional: tú decides qué ejecutar
**`limpiar_explicitamente(datos, columnas=None, indices=None, quitar_duplicados=False, subset=None)`**
- `datos`: tabla de entrada, que permanece intacta.
- `columnas`: nombres exactos que quieres retirar.
- `indices`: etiquetas de filas que has revisado y quieres retirar.
- `quitar_duplicados`: elimina duplicados solo si lo estableces en `True`.
- `subset`: columnas que definen un duplicado; `None` considera toda la fila.
- Retorna otra tabla. Nombres o índices inexistentes producen error.

Ejemplo: `limpiar_explicitamente(pd.DataFrame({"a": [1, 1, 2]}), quitar_duplicados=True)`.
Salida esperada: una tabla de dos filas, con índices `[0, 2]` y valores `[1, 2]`.

Las llamadas de limpieza están comentadas. Para activarlas, comenta la línea de continuidad
y descomenta la alternativa que asigna **el mismo nombre de salida**, sin ejecutar ambas.
No elimines observaciones de prueba para mejorar métricas. Si hay duplicados entre particiones,
resuelve la unidad de observación en origen y reinicia la división; no borres casos de prueba a posteriori.
En series no retires filas de entrenamiento: romperías la continuidad temporal.

In [ ]:
def limpiar_explicitamente(datos, columnas=None, indices=None, quitar_duplicados=False, subset=None):
    """Devuelve otra tabla al ejecutar únicamente las eliminaciones solicitadas."""
    etapa_columnas = datos.drop(columns=columnas or [])
    etapa_filas = etapa_columnas.drop(index=indices or [])
    etapa_duplicados = etapa_filas.drop_duplicates(subset=subset) if quitar_duplicados else etapa_filas.copy()
    return etapa_duplicados

df_train_limpio = df_reserva_train.copy()  # Continuidad sin eliminar datos
# df_train_limpio = limpiar_explicitamente(df_reserva_train, indices=[10, 25])
# df_train_limpio = limpiar_explicitamente(df_reserva_train, quitar_duplicados=True, subset=COLUMNAS_ID or None)
df_validacion_limpia = df_reserva_validacion.copy()
df_prueba_limpia = df_reserva_prueba.copy()

comparacion_etapas = pd.DataFrame({"Filas": [len(df_original), len(df_reglas), len(df_reserva_train), len(df_train_limpio)],
    "Columnas": [df_original.shape[1], df_reglas.shape[1], df_reserva_train.shape[1], df_train_limpio.shape[1]]},
    index=["Original", "Reglas explícitas", "Reserva entrenamiento", "Entrenamiento limpio"])
display(comparacion_etapas)
if df_train_limpio.empty:
    raise ValueError("La limpieza dejó entrenamiento vacío.")
if TIPO_PROBLEMA == "serie_tiempo" and not df_train_limpio.equals(df_reserva_train):
    raise ValueError("No elimines periodos en esta ruta temporal. Corrige el origen y reinicia.")

## 6. Dos funciones pequeñas para métricas reutilizables
**`metricas_regresion(reales, predicciones)`**: recibe vectores del mismo tamaño y orden.
Retorna un diccionario con MAE, RMSE, R², MAPE y cantidad de ceros excluidos del MAPE.
Ejemplo: `metricas_regresion([0, 2, 4], [0, 2, 4])` → MAE=0, RMSE=0, R²=1,
MAPE=0 y `ceros_excluidos_MAPE=1`. Si todos los reales son cero, MAPE es `NaN`.

**`metricas_clasificacion(reales, predicciones, probabilidades, clases, indice_positivo)`**:
recibe etiquetas numéricas codificadas, predicciones y una matriz de probabilidades cuyas columnas
siguen `clases`. `indice_positivo` identifica la clase positiva binaria; en multiclase se usa macro.
Retorna accuracy, precision, recall, F1 y AUC. AUC es `NaN` si no aparecen todas las clases en los reales.
Ejemplo ejecutable:
`metricas_clasificacion([0,1], [0,1], np.array([[.9,.1],[.1,.9]]), [0,1], 1)`
→ `{"Accuracy": 1, "Precision": 1, "Recall": 1, "F1": 1, "AUC": 1}`.

In [ ]:
def metricas_regresion(reales, predicciones):
    """Calcula errores; MAPE usa únicamente targets distintos de cero."""
    vector_real = np.asarray(reales, dtype=float)
    vector_predicho = np.asarray(predicciones, dtype=float)
    casos_no_cero = vector_real != 0
    return {
        "MAE": mean_absolute_error(vector_real, vector_predicho),
        "RMSE": np.sqrt(mean_squared_error(vector_real, vector_predicho)),
        "R2": r2_score(vector_real, vector_predicho) if len(vector_real) > 1 else np.nan,
        "MAPE_pct": np.mean(np.abs((vector_real[casos_no_cero] - vector_predicho[casos_no_cero]) / vector_real[casos_no_cero])) * 100 if casos_no_cero.any() else np.nan,
        "ceros_excluidos_MAPE": int((~casos_no_cero).sum())
    }

def metricas_clasificacion(reales, predicciones, probabilidades, clases, indice_positivo):
    """Métricas binarias o macro; respeta el orden declarado de probabilidades."""
    caso_binario = len(clases) == 2
    parametros_promedio = {"average": "binary", "pos_label": indice_positivo} if caso_binario else {"average": "macro", "labels": clases}
    auc_calculable = set(np.unique(reales)) == set(clases)
    valor_auc = (roc_auc_score(np.asarray(reales) == indice_positivo, probabilidades[:, list(clases).index(indice_positivo)])
        if caso_binario else roc_auc_score(reales, probabilidades, labels=clases, multi_class="ovr", average="macro")) if auc_calculable else np.nan
    return {"Accuracy": accuracy_score(reales, predicciones),
        "Precision": precision_score(reales, predicciones, zero_division=0, **parametros_promedio),
        "Recall": recall_score(reales, predicciones, zero_division=0, **parametros_promedio),
        "F1": f1_score(reales, predicciones, zero_division=0, **parametros_promedio), "AUC": valor_auc}

## 7. Preparación tabular: Pipeline explicado
Esta sección se ejecuta para clasificación y regresión. Las columnas se seleccionan por tus listas.
Si hay nulos y no activaste imputación, se detiene y te permite decidir una estrategia.

El Pipeline numérico imputa (si lo autorizas) y escala (si lo autorizas). El categórico imputa
y aplica one-hot encoding. `handle_unknown="ignore"` representa categorías nuevas con ceros;
elige `"error"` si prefieres detenerte. Esto no implica conocer la categoría ni garantizar precisión.
No se codifica cada archivo por separado.

Se usa una matriz densa para mantener legibilidad; con muchas categorías puede consumir mucha memoria.
Reduce predictores o adapta conscientemente el codificador para bases grandes.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    X_train = df_train_limpio.loc[:, predictores_declarados].copy()
    X_validacion = df_validacion_limpia.loc[:, predictores_declarados].copy()
    X_prueba = df_prueba_limpia.loc[:, predictores_declarados].copy()
    for nombre_revision, tabla_revision in [("train", X_train), ("validación", X_validacion), ("prueba", X_prueba)]:
        for columna_numerica in COLUMNAS_NUMERICAS:
            if not pd.api.types.is_numeric_dtype(tabla_revision[columna_numerica]):
                raise TypeError(f"{nombre_revision}: {columna_numerica} debe ser numérica.")
        if np.isinf(tabla_revision[COLUMNAS_NUMERICAS].to_numpy(dtype=float)).any():
            raise ValueError(f"{nombre_revision}: hay infinitos. Define un reemplazo explícito.")
        if not IMPUTAR_NUMERICAS and tabla_revision[COLUMNAS_NUMERICAS].isna().any().any():
            raise ValueError(f"{nombre_revision}: hay nulos numéricos. Decide si habilitas imputación.")
        if not IMPUTAR_CATEGORICAS and tabla_revision[COLUMNAS_CATEGORICAS].isna().any().any():
            raise ValueError(f"{nombre_revision}: hay nulos categóricos. Decide si habilitas imputación.")
        for columna_categorica in COLUMNAS_CATEGORICAS:
            tipos_categoria = {"texto" if isinstance(v, str) else "numero" if isinstance(v, (int, float, np.number, bool)) else "otro"
                for v in tabla_revision[columna_categorica].dropna().unique()}
            if "otro" in tipos_categoria or len(tipos_categoria) > 1:
                raise TypeError(f"{nombre_revision}: {columna_categorica} mezcla tipos incompatibles; declara una regla de texto.")
            categoria_train_numerica = pd.api.types.is_numeric_dtype(X_train[columna_categorica])
            if tipos_categoria and categoria_train_numerica != (tipos_categoria == {"numero"}):
                raise TypeError(f"{nombre_revision}: {columna_categorica} cambió entre códigos numéricos y texto.")
    pipeline_numerico = Pipeline([
        ("imputacion", SimpleImputer(strategy=ESTRATEGIA_NUMERICA, fill_value=VALOR_NUMERICO_CONSTANTE, keep_empty_features=True) if IMPUTAR_NUMERICAS else "passthrough"),
        ("escalado", StandardScaler() if ESCALAR else "passthrough")])
    pipeline_categorico = Pipeline([
        ("imputacion", SimpleImputer(strategy=ESTRATEGIA_CATEGORICA, fill_value=VALOR_CATEGORICO_CONSTANTE, keep_empty_features=True) if IMPUTAR_CATEGORICAS else "passthrough"),
        ("codificacion", OneHotEncoder(handle_unknown=CATEGORIA_DESCONOCIDA, sparse_output=False))])
    preprocesador_sin_ajustar = ColumnTransformer([
        ("numericas", pipeline_numerico, COLUMNAS_NUMERICAS),
        ("categoricas", pipeline_categorico, COLUMNAS_CATEGORICAS)], remainder="drop")

    if TIPO_PROBLEMA == "clasificacion":
        codificador_target_sin_ajustar = LabelEncoder()
        codificador_target_ajustado = copy.deepcopy(codificador_target_sin_ajustar).fit(df_train_limpio[TARGET])
        clases_originales = codificador_target_ajustado.classes_
        clases_codificadas = np.arange(len(clases_originales))
        if len(clases_originales) < 2:
            raise ValueError("La limpieza dejó menos de dos clases.")
        if len(clases_originales) == 2 and CLASE_POSITIVA not in clases_originales:
            raise ValueError("CLASE_POSITIVA no aparece en entrenamiento.")
        clases_fuera_train = (set(df_validacion_limpia[TARGET]) | set(df_prueba_limpia[TARGET])) - set(clases_originales)
        if clases_fuera_train:
            raise ValueError(f"La limpieza dejó clases fuera de entrenamiento: {clases_fuera_train}. Revisa la limpieza y partición.")
        indice_positivo = int(codificador_target_ajustado.transform([CLASE_POSITIVA])[0]) if len(clases_originales) == 2 else None
        y_train = pd.Series(codificador_target_ajustado.transform(df_train_limpio[TARGET]), index=X_train.index)
        y_validacion = pd.Series(codificador_target_ajustado.transform(df_validacion_limpia[TARGET]), index=X_validacion.index)
        display(pd.DataFrame({"Etiqueta original": clases_originales, "Código": clases_codificadas}))
    else:
        y_train = df_train_limpio[TARGET].copy()
        y_validacion = df_validacion_limpia[TARGET].copy()

## 8. Recorrido A · Clasificación
La línea base usa la clase más frecuente. Compara contra ella antes de atribuir valor al modelo.
Cada estimador tiene su Pipeline independiente; `clone(...).fit(...)` conserva el objeto sin ajustar.
Los modelos se entrenan con train; validación sirve para comparar y ajustar decisiones.

**Hiperparámetros:** `C` controla la regularización logística (menor C, mayor regularización);
`max_depth` limita la profundidad; `min_samples_leaf` exige ejemplos por hoja;
`n_estimators` controla árboles; `learning_rate` controla el tamaño de las correcciones en boosting;
`num_leaves` limita las hojas de LightGBM. El árbol sin límite permite observar el efecto del ajuste.
Los pesos de clase se mantienen sin cambios; puedes fijar `class_weight="balanced"` en los modelos
que lo admiten y comparar sus consecuencias sobre validación.

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    parametros_logistica = dict(C=1.0, max_iter=1500, random_state=SEMILLA)
    parametros_arbol_clasificacion = dict(max_depth=5, min_samples_leaf=5, random_state=SEMILLA)
    parametros_rf_clasificacion = dict(n_estimators=150, max_depth=10, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_clasificacion = dict(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=-1)
    parametros_lgb_clasificacion = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, min_child_samples=20, random_state=SEMILLA, n_jobs=-1, verbosity=-1)
    estimadores_clasificacion = {
        "Base frecuente": DummyClassifier(strategy="most_frequent"),
        "Regresión logística": LogisticRegression(**parametros_logistica),
        "Árbol sin límite": DecisionTreeClassifier(random_state=SEMILLA),
        "Árbol podado": DecisionTreeClassifier(**parametros_arbol_clasificacion),
        "Random Forest": RandomForestClassifier(**parametros_rf_clasificacion),
        "XGBoost": XGBClassifier(**parametros_xgb_clasificacion),
        "LightGBM": LGBMClassifier(**parametros_lgb_clasificacion)}
    pipelines_clasificacion_sin_ajustar = {nombre: Pipeline([
        ("preparacion", clone(preprocesador_sin_ajustar)), ("modelo", estimador)])
        for nombre, estimador in estimadores_clasificacion.items()}
    modelos_clasificacion_ajustados = {nombre: clone(pipeline).fit(X_train, y_train)
        for nombre, pipeline in pipelines_clasificacion_sin_ajustar.items()}

In [ ]:
if TIPO_PROBLEMA == "clasificacion":
    filas_metricas_clasificacion = []
    for nombre_clasificador, clasificador_ajustado in modelos_clasificacion_ajustados.items():
        for particion_clasificacion, datos_clasificacion, reales_clasificacion in [
            ("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]:
            probabilidades_clasificacion = clasificador_ajustado.predict_proba(datos_clasificacion)
            etiquetas_clasificacion = np.where(probabilidades_clasificacion[:, indice_positivo] >= UMBRAL,
                indice_positivo, 1 - indice_positivo) if len(clases_originales) == 2 else clasificador_ajustado.predict(datos_clasificacion)
            filas_metricas_clasificacion.append({"Modelo": nombre_clasificador, "Partición": particion_clasificacion,
                **metricas_clasificacion(reales_clasificacion, etiquetas_clasificacion, probabilidades_clasificacion, clases_codificadas, indice_positivo)})
    tabla_metricas_clasificacion = pd.DataFrame(filas_metricas_clasificacion).set_index(["Modelo", "Partición"])
    display(tabla_metricas_clasificacion.round(4))
    brechas_clasificacion = tabla_metricas_clasificacion.xs("Train", level="Partición") - tabla_metricas_clasificacion.xs("Validación", level="Partición")
    display(brechas_clasificacion.add_prefix("Train menos validación: ").round(4))

### 8.1 Tu elección y el punto de corte
Escribe un nombre de la tabla en `MODELO_CLASIFICACION_ELEGIDO`; el valor inicial es una elección
ilustrativa, no el ganador calculado. Cambia `UMBRAL` en configuración tras revisar el barrido de validación.
En multiclase se elige la clase de mayor probabilidad; no se aplica un umbral binario.
No vuelvas a elegir modelo después de mirar prueba.

In [ ]:
MODELO_CLASIFICACION_ELEGIDO = "Random Forest"
UMBRALES_A_COMPARAR = [0.20, 0.35, 0.50, 0.65, 0.80]
N_DECILES = 10

if TIPO_PROBLEMA == "clasificacion":
    if MODELO_CLASIFICACION_ELEGIDO not in modelos_clasificacion_ajustados:
        raise ValueError("Escribe un modelo de la tabla de clasificación.")
    modelo_clasificacion_elegido = modelos_clasificacion_ajustados[MODELO_CLASIFICACION_ELEGIDO]
    probabilidades_validacion_elegido = modelo_clasificacion_elegido.predict_proba(X_validacion)
    predicciones_validacion_elegido = np.where(probabilidades_validacion_elegido[:, indice_positivo] >= UMBRAL,
        indice_positivo, 1 - indice_positivo) if len(clases_originales) == 2 else modelo_clasificacion_elegido.predict(X_validacion)
    ConfusionMatrixDisplay.from_predictions(y_validacion, predicciones_validacion_elegido,
        labels=clases_codificadas, display_labels=clases_originales)
    plt.title("Matriz de confusión · validación")
    plt.show()
    print(classification_report(y_validacion, predicciones_validacion_elegido,
        labels=clases_codificadas, target_names=[str(c) for c in clases_originales], zero_division=0))
    if len(clases_originales) == 2:
        filas_umbrales = [{"Umbral": u, **metricas_clasificacion(y_validacion,
            np.where(probabilidades_validacion_elegido[:, indice_positivo] >= u, indice_positivo, 1 - indice_positivo),
            probabilidades_validacion_elegido, clases_codificadas, indice_positivo)} for u in UMBRALES_A_COMPARAR]
        display(pd.DataFrame(filas_umbrales).round(4))
        fpr_validacion, tpr_validacion, cortes_roc = roc_curve(y_validacion == indice_positivo, probabilidades_validacion_elegido[:, indice_positivo])
        plt.plot(fpr_validacion, tpr_validacion, label=MODELO_CLASIFICACION_ELEGIDO)
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.xlabel("Tasa de falsos positivos")
        plt.ylabel("Recall de la clase positiva")
        plt.legend()
        plt.show()
        base_deciles = pd.DataFrame({"probabilidad": probabilidades_validacion_elegido[:, indice_positivo],
            "evento": (y_validacion.to_numpy() == indice_positivo).astype(int)})
        if len(base_deciles) >= N_DECILES and base_deciles["evento"].sum() > 0:
            base_deciles_ordenada = base_deciles.sort_values("probabilidad", ascending=False, kind="stable")
            base_deciles_grupos = base_deciles_ordenada.assign(decil=np.floor(np.arange(len(base_deciles_ordenada)) * N_DECILES / len(base_deciles_ordenada)).astype(int) + 1)
            resumen_deciles = base_deciles_grupos.groupby("decil").agg(casos=("evento", "size"), eventos=("evento", "sum"), tasa=("evento", "mean"), prob_min=("probabilidad", "min"), prob_max=("probabilidad", "max"))
            tabla_deciles = resumen_deciles.assign(lift=resumen_deciles["tasa"] / base_deciles["evento"].mean(),
                captura_acumulada_pct=resumen_deciles["eventos"].cumsum() / base_deciles["evento"].sum() * 100)
            display(tabla_deciles)
        else:
            print("Deciles omitidos: necesitas suficientes filas y al menos un evento.")

## 9. Recorrido B · Regresión
La línea base predice el promedio de entrenamiento. Compara errores en unidades del target (MAE y RMSE).
R² puede ser negativo. MAPE es inestable cerca de cero y aquí excluye ceros, indicando cuántos.

Edita los hiperparámetros junto a cada modelo. `max_depth`, `min_samples_leaf`, `n_estimators`,
`learning_rate` y `num_leaves` tienen el mismo papel descrito en clasificación.
La regresión lineal se ajusta sin regularización; no necesita un parámetro C.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    parametros_arbol_regresion = dict(max_depth=5, min_samples_leaf=5, random_state=SEMILLA)
    parametros_rf_regresion = dict(n_estimators=150, max_depth=10, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_regresion = dict(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=-1)
    parametros_lgb_regresion = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, min_child_samples=20, random_state=SEMILLA, n_jobs=-1, verbosity=-1)
    estimadores_regresion = {
        "Base promedio": DummyRegressor(strategy="mean"),
        "Regresión lineal": LinearRegression(),
        "Árbol sin límite": DecisionTreeRegressor(random_state=SEMILLA),
        "Árbol podado": DecisionTreeRegressor(**parametros_arbol_regresion),
        "Random Forest": RandomForestRegressor(**parametros_rf_regresion),
        "XGBoost": XGBRegressor(**parametros_xgb_regresion),
        "LightGBM": LGBMRegressor(**parametros_lgb_regresion)}
    pipelines_regresion_sin_ajustar = {nombre: Pipeline([
        ("preparacion", clone(preprocesador_sin_ajustar)), ("modelo", estimador)])
        for nombre, estimador in estimadores_regresion.items()}
    modelos_regresion_ajustados = {nombre: clone(pipeline).fit(X_train, y_train)
        for nombre, pipeline in pipelines_regresion_sin_ajustar.items()}
    filas_metricas_regresion = [{"Modelo": nombre, "Partición": particion,
        **metricas_regresion(reales, modelo.predict(datos))}
        for nombre, modelo in modelos_regresion_ajustados.items()
        for particion, datos, reales in [("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]]
    tabla_metricas_regresion = pd.DataFrame(filas_metricas_regresion).set_index(["Modelo", "Partición"])
    display(tabla_metricas_regresion.round(4))
    brechas_regresion = tabla_metricas_regresion.xs("Validación", level="Partición") - tabla_metricas_regresion.xs("Train", level="Partición")
    display(brechas_regresion[["MAE", "RMSE"]].add_prefix("Validación menos train: "))

In [ ]:
MODELO_REGRESION_ELEGIDO = "Random Forest"
COLUMNAS_SEGMENTOS = []  # Ejemplo: ["canal"]

if TIPO_PROBLEMA == "regresion":
    if MODELO_REGRESION_ELEGIDO not in modelos_regresion_ajustados:
        raise ValueError("Escribe un modelo de la tabla de regresión.")
    modelo_regresion_elegido = modelos_regresion_ajustados[MODELO_REGRESION_ELEGIDO]
    predicciones_validacion_regresion = modelo_regresion_elegido.predict(X_validacion)
    diagnostico_regresion = df_validacion_limpia.assign(prediccion=predicciones_validacion_regresion,
        residuo=y_validacion.to_numpy() - predicciones_validacion_regresion,
        error_absoluto=np.abs(y_validacion.to_numpy() - predicciones_validacion_regresion))
    fig_errores, ejes_errores = plt.subplots(1, 3, figsize=(15, 4))
    ejes_errores[0].scatter(y_validacion, predicciones_validacion_regresion, alpha=0.5)
    limites_reales = [y_validacion.min(), y_validacion.max()]
    ejes_errores[0].plot(limites_reales, limites_reales, "--", color="gray")
    ejes_errores[0].set(xlabel="Real", ylabel="Predicción")
    ejes_errores[1].scatter(diagnostico_regresion["prediccion"], diagnostico_regresion["residuo"], alpha=0.5)
    ejes_errores[1].axhline(0, color="gray", linestyle="--")
    ejes_errores[1].set(xlabel="Predicción", ylabel="Residuo: real − predicción")
    sns.histplot(diagnostico_regresion["residuo"], ax=ejes_errores[2])
    plt.tight_layout()
    plt.show()
    display(diagnostico_regresion.sort_values("error_absoluto", ascending=False).head(10))
    for segmento_error in COLUMNAS_SEGMENTOS:
        display(diagnostico_regresion.groupby(segmento_error, dropna=False)["error_absoluto"].agg(["count", "mean", "median"]))

## 10. Opcional · Búsqueda de hiperparámetros tabulares
Inicialmente desactivada. RandomizedSearchCV muestrea tu lista; GridSearchCV recorre todas las combinaciones.
Ambas usan solo entrenamiento y ajustan la preparación dentro de cada fold.
El prefijo `modelo__` identifica parámetros del estimador dentro del Pipeline.

Se muestra `cv_results_`, pero ningún resultado reemplaza tu modelo. Si quieres adoptarlo,
copia los parámetros elegidos a la celda del modelo, vuelve a entrenar y compara en validación.
Esta sección busca Random Forest. Puedes cambiar el estimador y su espacio de parámetros juntos.
Para series utiliza el backtesting recursivo de la sección siguiente, no CV aleatoria.

In [ ]:
EJECUTAR_RANDOM_SEARCH = False
EJECUTAR_GRID_SEARCH = False
FOLDS_CV = 3
ITERACIONES_RANDOM = 4
GRILLA_RF = {"modelo__n_estimators": [80, 150], "modelo__max_depth": [4, 8], "modelo__min_samples_leaf": [3, 6]}

if TIPO_PROBLEMA in {"clasificacion", "regresion"} and (EJECUTAR_RANDOM_SEARCH or EJECUTAR_GRID_SEARCH):
    if TIPO_PROBLEMA == "clasificacion":
        if y_train.value_counts().min() < FOLDS_CV:
            raise ValueError("No hay suficientes filas por clase para FOLDS_CV.")
        pipeline_busqueda = pipelines_clasificacion_sin_ajustar["Random Forest"]
        cv_busqueda = StratifiedKFold(n_splits=FOLDS_CV, shuffle=True, random_state=SEMILLA)
        metrica_busqueda = "f1_macro"
    else:
        pipeline_busqueda = pipelines_regresion_sin_ajustar["Random Forest"]
        cv_busqueda = KFold(n_splits=FOLDS_CV, shuffle=True, random_state=SEMILLA)
        metrica_busqueda = "neg_mean_absolute_error"  # Mayor = menor MAE
    if EJECUTAR_RANDOM_SEARCH:
        busqueda_random_sin_ajustar = RandomizedSearchCV(clone(pipeline_busqueda), GRILLA_RF,
            n_iter=ITERACIONES_RANDOM, scoring=metrica_busqueda, cv=cv_busqueda, random_state=SEMILLA, n_jobs=-1, refit=False, error_score="raise")
        busqueda_random_ajustada = copy.deepcopy(busqueda_random_sin_ajustar).fit(X_train, y_train)
        resultados_random = pd.DataFrame(busqueda_random_ajustada.cv_results_)
        display(resultados_random[["params", "mean_test_score", "std_test_score", "rank_test_score"]])
    if EJECUTAR_GRID_SEARCH:
        busqueda_grid_sin_ajustar = GridSearchCV(clone(pipeline_busqueda), GRILLA_RF,
            scoring=metrica_busqueda, cv=cv_busqueda, n_jobs=-1, refit=False, error_score="raise")
        busqueda_grid_ajustada = copy.deepcopy(busqueda_grid_sin_ajustar).fit(X_train, y_train)
        resultados_grid = pd.DataFrame(busqueda_grid_ajustada.cv_results_)
        display(resultados_grid[["params", "mean_test_score", "std_test_score", "rank_test_score"]])

## 11. Recorrido C · Series de tiempo
Este recorrido aprende de una serie regular y univariante: fecha y target. No usa variables externas.
`HORIZONTE` controla el pronóstico futuro; validación y prueba usan toda la ventana reservada.

**`construir_rezagos(historia, rezagos, ventanas)`**
- `historia`: Series numérica con índice DatetimeIndex regular y ordenado.
- `rezagos`: lista de periodos anteriores, por ejemplo `[1, 7]`.
- `ventanas`: tamaños de medias móviles, por ejemplo `[7]`.
- Devuelve otra tabla con calendario, `lag_k`, `media_k` y `objetivo`.
  Las medias usan `shift(1)`: nunca incluyen el target de la misma fila.
- Ejemplo ejecutable: `construir_rezagos(pd.Series([10,20,30], index=pd.date_range("2025-01-01", periods=3)), [1], [2]).iloc[-1]`.
  La tercera fila tiene `lag_1=20`, `media_2=15` y `objetivo=30`.

**`pronostico_recursivo(modelo, historia, fechas, rezagos, ventanas)`**
- `modelo`: regresor ajustado o `None` para la línea base de persistencia.
- `historia`: Series regular que contiene exclusivamente valores conocidos antes del horizonte.
- `fechas`: DatetimeIndex de los siguientes periodos consecutivos, con frecuencia establecida.
- `rezagos` / `ventanas`: la misma configuración del entrenamiento.
- Devuelve una Series de predicciones; cada predicción alimenta la historia del siguiente paso.
- Ejemplo ejecutable: `pronostico_recursivo(None, pd.Series([10,20,30], index=pd.date_range("2025-01-01", periods=3)), pd.date_range("2025-01-04", periods=2, freq="D"), [1], [2]).tolist()` → `[30.0,30.0]`.
La historia de entrada no cambia. La lista interna de valores del pronóstico crece de forma local;
no se sobrescribe el DataFrame histórico ni se consultan valores reales futuros.

In [ ]:
def construir_rezagos(historia, rezagos, ventanas):
    """Construye otra tabla; cada predictor histórico usa solo periodos anteriores."""
    tabla_calendario = pd.DataFrame({"anio": historia.index.year, "mes": historia.index.month,
        "dia": historia.index.day, "dia_semana": historia.index.dayofweek}, index=historia.index)
    tabla_memoria = tabla_calendario.assign(**{
        **{f"lag_{k}": historia.shift(k) for k in rezagos},
        **{f"media_{k}": historia.shift(1).rolling(k, min_periods=k).mean() for k in ventanas}})
    tabla_objetivo = tabla_memoria.assign(objetivo=historia.to_numpy())
    return tabla_objetivo

def pronostico_recursivo(modelo, historia, fechas, rezagos, ventanas):
    """Pronostica sin leer el target futuro; modelo=None representa persistencia."""
    if len(historia) < max(rezagos + ventanas):
        raise ValueError("Historia insuficiente para construir la memoria temporal.")
    if len(fechas) == 0 or fechas.freq is None:
        raise ValueError("fechas debe ser un DatetimeIndex no vacío con frecuencia explícita.")
    esperadas_recursion = pd.date_range(historia.index[-1] + fechas.freq, periods=len(fechas), freq=fechas.freq)
    if not fechas.equals(esperadas_recursion):
        raise ValueError("El pronóstico debe comenzar en el siguiente periodo y ser consecutivo.")
    valores_conocidos = historia.to_numpy(dtype=float).tolist()
    predicciones_recursivas = []
    for fecha_paso in fechas:
        fila_paso = pd.DataFrame({"anio": [fecha_paso.year], "mes": [fecha_paso.month],
            "dia": [fecha_paso.day], "dia_semana": [fecha_paso.dayofweek],
            **{f"lag_{k}": [valores_conocidos[-k]] for k in rezagos},
            **{f"media_{k}": [np.mean(valores_conocidos[-k:])] for k in ventanas}}, index=[fecha_paso])
        prediccion_paso = float(valores_conocidos[-1]) if modelo is None else float(modelo.predict(fila_paso)[0])
        predicciones_recursivas.append(prediccion_paso)
        valores_conocidos.append(prediccion_paso)
    return pd.Series(predicciones_recursivas, index=fechas, name="prediccion")

### 11.1 Entrenar y comparar con validación recursiva
Se excluyen únicamente las primeras filas sin historia suficiente para construir rezagos;
no son atípicos ni filas eliminadas por limpieza. Se informa cuántas quedan fuera.
Train se evalúa a un paso con rezagos reales; validación se evalúa recursivamente.
La diferencia incluye tanto generalización como acumulación de errores y no mide solo sobreajuste.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    historia_train = df_train_limpio.set_index(FECHA)[TARGET].copy()
    tabla_temporal_completa = construir_rezagos(historia_train, REZAGOS, VENTANAS)
    tabla_temporal_entrenable = tabla_temporal_completa.dropna()
    print("Filas iniciales sin memoria suficiente:", len(tabla_temporal_completa) - len(tabla_temporal_entrenable))
    X_train_temporal = tabla_temporal_entrenable.drop(columns="objetivo")
    y_train_temporal = tabla_temporal_entrenable["objetivo"].copy()
    parametros_rf_temporal = dict(n_estimators=150, max_depth=8, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    parametros_xgb_temporal = dict(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=SEMILLA, n_jobs=-1)
    modelos_temporales_sin_ajustar = {"Random Forest": RandomForestRegressor(**parametros_rf_temporal),
        "XGBoost": XGBRegressor(**parametros_xgb_temporal)}
    modelos_temporales_ajustados = {nombre: clone(modelo).fit(X_train_temporal, y_train_temporal)
        for nombre, modelo in modelos_temporales_sin_ajustar.items()}
    fechas_validacion_temporal = pd.date_range(df_validacion_limpia[FECHA].iloc[0], periods=len(df_validacion_limpia), freq=FRECUENCIA)
    predicciones_validacion_temporal = {nombre: pronostico_recursivo(modelo, historia_train, fechas_validacion_temporal, REZAGOS, VENTANAS)
        for nombre, modelo in {"Persistencia": None, **modelos_temporales_ajustados}.items()}
    filas_metricas_temporales = [
        {"Modelo": nombre, "Partición": "Validación recursiva", **metricas_regresion(df_validacion_limpia[TARGET], prediccion)}
        for nombre, prediccion in predicciones_validacion_temporal.items()]
    filas_metricas_train_temporal = [{"Modelo": nombre, "Partición": "Train un paso",
        **metricas_regresion(y_train_temporal, modelo.predict(X_train_temporal))}
        for nombre, modelo in modelos_temporales_ajustados.items()]
    tabla_metricas_temporales = pd.DataFrame(filas_metricas_train_temporal + filas_metricas_temporales).set_index(["Modelo", "Partición"])
    display(tabla_metricas_temporales)
    comparacion_temporal = pd.DataFrame({"Real": df_validacion_limpia[TARGET].to_numpy(),
        **{nombre: pred.to_numpy() for nombre, pred in predicciones_validacion_temporal.items()}}, index=fechas_validacion_temporal)
    comparacion_temporal.plot(figsize=(12, 4), title="Validación: horizonte completo sin targets futuros")
    plt.show()

### 11.2 Backtesting y selección manual
Las ventanas terminan dentro de entrenamiento: cada modelo usa solo la historia previa a su ventana.
`PASO_BACKTEST` controla separación entre orígenes y `HORIZONTE` su longitud. Pueden solaparse;
si quieres ventanas disjuntas, usa paso igual o mayor al horizonte.
No se ajustan hiperparámetros automáticamente. Cambia sus diccionarios y compara de nuevo.

In [ ]:
MODELO_TEMPORAL_ELEGIDO = "Random Forest"  # También "XGBoost" o "Persistencia"

if TIPO_PROBLEMA == "serie_tiempo":
    origenes_backtest = [len(historia_train) - HORIZONTE - PASO_BACKTEST * k for k in reversed(range(VENTANAS_BACKTEST))]
    if min(origenes_backtest) <= memoria_temporal + 5:
        raise ValueError("Reduce ventanas/horizonte/paso de backtesting o aporta más historia.")
    filas_backtest = []
    for numero_ventana, origen_ventana in enumerate(origenes_backtest, start=1):
        historia_ventana = historia_train.iloc[:origen_ventana].copy()
        tabla_ventana_completa = construir_rezagos(historia_ventana, REZAGOS, VENTANAS)
        tabla_ventana_entrenable = tabla_ventana_completa.dropna()
        fechas_ventana = pd.date_range(historia_train.index[origen_ventana], periods=HORIZONTE, freq=FRECUENCIA)
        reales_ventana = historia_train.loc[fechas_ventana]
        for nombre_backtest, modelo_backtest in {"Persistencia": None, **modelos_temporales_sin_ajustar}.items():
            modelo_ventana_ajustado = clone(modelo_backtest).fit(tabla_ventana_entrenable.drop(columns="objetivo"), tabla_ventana_entrenable["objetivo"]) if modelo_backtest is not None else None
            prediccion_ventana = pronostico_recursivo(modelo_ventana_ajustado, historia_ventana, fechas_ventana, REZAGOS, VENTANAS)
            filas_backtest.append({"Ventana": numero_ventana, "Modelo": nombre_backtest, "Inicio": fechas_ventana[0],
                **metricas_regresion(reales_ventana, prediccion_ventana)})
    tabla_backtest = pd.DataFrame(filas_backtest)
    display(tabla_backtest)
    display(tabla_backtest.groupby("Modelo")[["MAE", "RMSE", "MAPE_pct"]].mean())
    tabla_backtest.pivot(index="Ventana", columns="Modelo", values="MAE").plot.bar(title="MAE por ventana de backtesting")
    plt.show()
    if MODELO_TEMPORAL_ELEGIDO not in {"Persistencia", *modelos_temporales_ajustados}:
        raise ValueError("Elige un modelo temporal de la comparación.")

## 12. Evaluación final en prueba: ejecutar después de elegir
Esta celda no selecciona modelos ni cambia parámetros. El modelo elegido se reentrena con
train + validación y recibe **otro nombre**; el modelo usado durante la comparación se conserva.
No ajustes decisiones a partir de esta tabla: hacerlo convertiría prueba en otra validación.

En series el modelo final conoce hasta el final de validación y pronostica toda prueba recursivamente.
En tabular el Pipeline se vuelve a ajustar solo con train + validación.

In [ ]:
if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
    X_desarrollo_final = pd.concat([X_train, X_validacion], axis=0)
    y_desarrollo_final = pd.concat([y_train, y_validacion], axis=0)
    pipeline_final_sin_ajustar = clone(modelo_clasificacion_elegido if TIPO_PROBLEMA == "clasificacion" else modelo_regresion_elegido)
    pipeline_final_ajustado = clone(pipeline_final_sin_ajustar).fit(X_desarrollo_final, y_desarrollo_final)
    if TIPO_PROBLEMA == "clasificacion":
        y_prueba = pd.Series(codificador_target_ajustado.transform(df_prueba_limpia[TARGET]), index=X_prueba.index)
        probabilidades_prueba = pipeline_final_ajustado.predict_proba(X_prueba)
        etiquetas_prueba = np.where(probabilidades_prueba[:, indice_positivo] >= UMBRAL, indice_positivo,
            1 - indice_positivo) if len(clases_originales) == 2 else pipeline_final_ajustado.predict(X_prueba)
        metricas_prueba_final = metricas_clasificacion(y_prueba, etiquetas_prueba, probabilidades_prueba, clases_codificadas, indice_positivo)
        ConfusionMatrixDisplay.from_predictions(y_prueba, etiquetas_prueba, labels=clases_codificadas, display_labels=clases_originales)
        plt.title("Prueba final")
        plt.show()
    else:
        predicciones_prueba = pipeline_final_ajustado.predict(X_prueba)
        metricas_prueba_final = metricas_regresion(df_prueba_limpia[TARGET], predicciones_prueba)
else:
    df_desarrollo_temporal = pd.concat([df_train_limpio, df_validacion_limpia], axis=0)
    historia_desarrollo_temporal = df_desarrollo_temporal.set_index(FECHA)[TARGET].copy()
    tabla_desarrollo_temporal_completa = construir_rezagos(historia_desarrollo_temporal, REZAGOS, VENTANAS)
    tabla_desarrollo_temporal_entrenable = tabla_desarrollo_temporal_completa.dropna()
    modelo_temporal_final_ajustado = clone(modelos_temporales_sin_ajustar[MODELO_TEMPORAL_ELEGIDO]).fit(
        tabla_desarrollo_temporal_entrenable.drop(columns="objetivo"), tabla_desarrollo_temporal_entrenable["objetivo"]) if MODELO_TEMPORAL_ELEGIDO != "Persistencia" else None
    fechas_prueba_temporal = pd.date_range(df_prueba_limpia[FECHA].iloc[0], periods=len(df_prueba_limpia), freq=FRECUENCIA)
    predicciones_prueba_temporal = pronostico_recursivo(modelo_temporal_final_ajustado, historia_desarrollo_temporal, fechas_prueba_temporal, REZAGOS, VENTANAS)
    metricas_prueba_final = metricas_regresion(df_prueba_limpia[TARGET], predicciones_prueba_temporal)
    diagnostico_prueba_temporal = pd.DataFrame({"Real": df_prueba_limpia[TARGET].to_numpy(), "Predicción": predicciones_prueba_temporal.to_numpy()}, index=fechas_prueba_temporal)
    diagnostico_prueba_temporal.plot(figsize=(12, 4), title="Prueba recursiva final")
    plt.show()
display(pd.Series(metricas_prueba_final, name="Prueba final").to_frame())

## 13. Opcional · Importancia de variables y reglas
Desactivada inicialmente. Permutation importance mide cuánto empeora una métrica al permutar
un predictor de validación. Variables correlacionadas pueden repartirse o esconder importancia.
En clasificación/regresión se calcula sobre el modelo comparado, que no vio validación.
Si eliges un árbol puedes visualizar sus reglas con `plot_tree`.
Para series se muestra la importancia interna del modelo elegido; no tiene interpretación causal.

In [ ]:
EJECUTAR_IMPORTANCIA = False
MOSTRAR_ARBOL = False
TOP_IMPORTANCIAS = 15

if EJECUTAR_IMPORTANCIA:
    if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
        modelo_para_importancia = modelo_clasificacion_elegido if TIPO_PROBLEMA == "clasificacion" else modelo_regresion_elegido
        resultado_permutacion = permutation_importance(modelo_para_importancia, X_validacion, y_validacion,
            scoring="f1_macro" if TIPO_PROBLEMA == "clasificacion" else "neg_mean_absolute_error", n_repeats=5, random_state=SEMILLA, n_jobs=-1)
        tabla_importancias = pd.DataFrame({"variable": predictores_declarados, "importancia": resultado_permutacion.importances_mean,
            "desviacion": resultado_permutacion.importances_std}).sort_values("importancia", ascending=False)
        if MOSTRAR_ARBOL and isinstance(modelo_para_importancia.named_steps["modelo"], (DecisionTreeClassifier, DecisionTreeRegressor)):
            plt.figure(figsize=(18, 7))
            plot_tree(modelo_para_importancia.named_steps["modelo"], max_depth=3,
                feature_names=modelo_para_importancia.named_steps["preparacion"].get_feature_names_out(), filled=True)
            plt.show()
    elif MODELO_TEMPORAL_ELEGIDO != "Persistencia":
        tabla_importancias = pd.DataFrame({"variable": X_train_temporal.columns,
            "importancia": modelos_temporales_ajustados[MODELO_TEMPORAL_ELEGIDO].feature_importances_}).sort_values("importancia", ascending=False)
    else:
        tabla_importancias = pd.DataFrame(columns=["variable", "importancia"])
        print("Persistencia no tiene importancias de árboles.")
    display(tabla_importancias.head(TOP_IMPORTANCIAS))
    if not tabla_importancias.empty:
        tabla_importancias.head(TOP_IMPORTANCIAS).set_index("variable")["importancia"].sort_values().plot.barh()
        plt.show()

### 13.1 SHAP opcional, con muestra pequeña y algoritmo explícito
El explicador por permutación trabaja sobre los predictores ya transformados del modelo comparado.
El fondo proviene de train. En clasificación explica probabilidades; en regresión, unidades del target.
Las categorías aparecen como columnas one-hot. No son efectos causales.

`CLASE_SHAP` es la etiqueta original cuya probabilidad quieres explicar.
En series se explican predicciones **a un paso**, no el efecto total del pronóstico recursivo.
Esta celda no se utiliza para elegir el modelo después de mirar prueba.

In [ ]:
EJECUTAR_SHAP = False
FILAS_FONDO_SHAP = 30
FILAS_EXPLICAR_SHAP = 5
CLASE_SHAP = CLASE_POSITIVA

if EJECUTAR_SHAP:
    import shap
    if TIPO_PROBLEMA in {"clasificacion", "regresion"}:
        pipeline_para_shap = modelo_clasificacion_elegido if TIPO_PROBLEMA == "clasificacion" else modelo_regresion_elegido
        fondo_shap = pipeline_para_shap.named_steps["preparacion"].transform(X_train.head(FILAS_FONDO_SHAP))
        datos_shap = pipeline_para_shap.named_steps["preparacion"].transform(X_validacion.head(FILAS_EXPLICAR_SHAP))
        nombres_shap = pipeline_para_shap.named_steps["preparacion"].get_feature_names_out()
        funcion_prediccion_shap = pipeline_para_shap.named_steps["modelo"].predict_proba if TIPO_PROBLEMA == "clasificacion" else pipeline_para_shap.named_steps["modelo"].predict
        explicador_shap = shap.Explainer(funcion_prediccion_shap, fondo_shap, algorithm="permutation", feature_names=nombres_shap, seed=SEMILLA)
        valores_shap = explicador_shap(datos_shap, max_evals=2 * len(nombres_shap) + 1)
        if TIPO_PROBLEMA == "clasificacion":
            indice_clase_shap = int(codificador_target_ajustado.transform([CLASE_SHAP])[0])
            valores_shap_clase = valores_shap[:, :, indice_clase_shap]
            shap.plots.bar(valores_shap_clase)
            shap.plots.waterfall(valores_shap_clase[0])
        else:
            shap.plots.bar(valores_shap)
            shap.plots.waterfall(valores_shap[0])
    elif MODELO_TEMPORAL_ELEGIDO != "Persistencia":
        modelo_shap_temporal = modelos_temporales_ajustados[MODELO_TEMPORAL_ELEGIDO]
        explicador_temporal_shap = shap.Explainer(modelo_shap_temporal.predict, X_train_temporal.head(FILAS_FONDO_SHAP), algorithm="permutation", seed=SEMILLA)
        valores_temporales_shap = explicador_temporal_shap(X_train_temporal.tail(FILAS_EXPLICAR_SHAP), max_evals=2 * X_train_temporal.shape[1] + 1)
        shap.plots.bar(valores_temporales_shap)
        shap.plots.waterfall(valores_temporales_shap[0])
    else:
        print("SHAP omitido para la línea base de persistencia.")

## 14. Guardar modelo, esquema y configuración
El archivo `.joblib` conserva el modelo ajustado, codificador del target, reglas y columnas.
El JSON sirve para consultar configuración y versiones. Carga solo archivos joblib de tu confianza.

En series, `REENTRENAR_SERIE_CON_TODA_HISTORIA=True` autoriza crear otro modelo operativo después de
evaluar prueba, usando toda la historia disponible. No cambia las métricas ya reportadas.
Si lo desactivas, el origen operativo queda al final de validación y el pronóstico deberá comenzar allí.
Para tabular se guarda el Pipeline final entrenado con train + validación; prueba queda fuera.

In [ ]:
DIRECTORIO_SALIDA = "/content/salidas_guia_pc2"
REENTRENAR_SERIE_CON_TODA_HISTORIA = True

directorio_salida = Path(DIRECTORIO_SALIDA)
directorio_salida.mkdir(parents=True, exist_ok=True)
configuracion_modelo = {
    "tipo_problema": TIPO_PROBLEMA, "target": TARGET, "numericas": COLUMNAS_NUMERICAS,
    "categoricas": COLUMNAS_CATEGORICAS, "predictores": predictores_declarados,
    "identificadores": COLUMNAS_ID, "exclusiones": COLUMNAS_EXCLUIR,
    "reglas_numericas": REGLAS_NUMERICAS, "reglas_textos": REGLAS_TEXTOS,
    "reglas_fechas": REGLAS_FECHAS, "reglas_reemplazos": REGLAS_REEMPLAZOS,
    "imputar_numericas": IMPUTAR_NUMERICAS, "imputar_categoricas": IMPUTAR_CATEGORICAS,
    "estrategia_numerica": ESTRATEGIA_NUMERICA, "estrategia_categorica": ESTRATEGIA_CATEGORICA,
    "escalar": ESCALAR, "categoria_desconocida": CATEGORIA_DESCONOCIDA,
    "umbral": UMBRAL, "clase_positiva": CLASE_POSITIVA,
    "fecha": FECHA, "formato_fecha": FORMATO_FECHA_SERIE, "frecuencia": FRECUENCIA,
    "rezagos": REZAGOS, "ventanas": VENTANAS, "horizonte": HORIZONTE,
    "semilla": SEMILLA, "fraccion_validacion": FRACCION_VALIDACION, "fraccion_prueba": FRACCION_PRUEBA,
    "versiones": versiones_entorno}

if TIPO_PROBLEMA == "serie_tiempo":
    df_historia_operativa = pd.concat([df_train_limpio, df_validacion_limpia, df_prueba_limpia], axis=0) if REENTRENAR_SERIE_CON_TODA_HISTORIA else df_desarrollo_temporal.copy()
    historia_operativa = df_historia_operativa.set_index(FECHA)[TARGET].copy()
    tabla_operativa_completa = construir_rezagos(historia_operativa, REZAGOS, VENTANAS)
    tabla_operativa_entrenable = tabla_operativa_completa.dropna()
    modelo_temporal_operativo = clone(modelos_temporales_sin_ajustar[MODELO_TEMPORAL_ELEGIDO]).fit(
        tabla_operativa_entrenable.drop(columns="objetivo"), tabla_operativa_entrenable["objetivo"]) if MODELO_TEMPORAL_ELEGIDO != "Persistencia" else None
    paquete_modelo = {"configuracion": configuracion_modelo, "modelo": modelo_temporal_operativo,
        "historia": historia_operativa, "nombre_modelo": MODELO_TEMPORAL_ELEGIDO,
        "esquema": {"fecha": str(df_historia_operativa[FECHA].dtype), "target": str(historia_operativa.dtype)}}
else:
    paquete_modelo = {"configuracion": configuracion_modelo, "modelo": pipeline_final_ajustado,
        "codificador_target": codificador_target_ajustado if TIPO_PROBLEMA == "clasificacion" else None,
        "nombre_modelo": MODELO_CLASIFICACION_ELEGIDO if TIPO_PROBLEMA == "clasificacion" else MODELO_REGRESION_ELEGIDO,
        "esquema": X_desarrollo_final.dtypes.astype(str).to_dict()}

ruta_modelo_guardado = directorio_salida / "modelo_y_configuracion.joblib"
ruta_configuracion_guardada = directorio_salida / "configuracion.json"
joblib.dump(paquete_modelo, ruta_modelo_guardado)
ruta_configuracion_guardada.write_text(json.dumps({"configuracion": configuracion_modelo,
    "esquema": paquete_modelo["esquema"], "modelo": paquete_modelo["nombre_modelo"]}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Modelo:", ruta_modelo_guardado)
print("Configuración:", ruta_configuracion_guardada)

## 15. Prueba con una base NUEVA
Esta sección usa el paquete **recargado desde disco**, para demostrar que no necesita reajustar nada.
El target es opcional. Mantén los nombres y unidades de los predictores usados al entrenar.
Las filas se conservan en el orden del archivo. Las columnas adicionales se informan y no se modelan.

Para tabular carga un CSV/Excel distinto. Para series puedes cargar un archivo futuro con columna fecha,
o usar `USAR_ARCHIVO_FUTURO=False` para construir explícitamente las siguientes `HORIZONTE` fechas.
El archivo temporal debe contener periodos consecutivos desde el origen del modelo guardado; no se
reordena ni agrega automáticamente. Esta versión no requiere ni utiliza variables externas.

Las columnas de resultados comienzan con `ml_`; evita ese prefijo en tu archivo para prevenir colisiones.

In [ ]:
SUBIR_BASE_NUEVA = False
RUTA_BASE_NUEVA = "/content/mi_base_nueva.csv"
FORMATO_BASE_NUEVA = "csv"
SEPARADOR_NUEVA = ","
CODIFICACION_NUEVA = "utf-8"
DECIMAL_NUEVA = "."
HOJA_NUEVA = 0
EVALUAR_BASE_NUEVA_CON_TARGET = False
USAR_ARCHIVO_FUTURO = False

In [ ]:
paquete_modelo_cargado = joblib.load(ruta_modelo_guardado)
configuracion_cargada = paquete_modelo_cargado["configuracion"]
modelo_cargado = paquete_modelo_cargado["modelo"]
tipo_cargado = configuracion_cargada["tipo_problema"]
target_cargado = configuracion_cargada["target"]

if tipo_cargado == "serie_tiempo" and not USAR_ARCHIVO_FUTURO:
    fechas_futuras_configuradas = pd.date_range(
        paquete_modelo_cargado["historia"].index[-1] + pd.tseries.frequencies.to_offset(configuracion_cargada["frecuencia"]),
        periods=configuracion_cargada["horizonte"], freq=configuracion_cargada["frecuencia"])
    df_nueva_original = pd.DataFrame({configuracion_cargada["fecha"]: fechas_futuras_configuradas})
else:
    if SUBIR_BASE_NUEVA:
        from google.colab import files
        archivos_nuevos_subidos = files.upload()
        if len(archivos_nuevos_subidos) != 1:
            raise ValueError("Sube exactamente un archivo nuevo.")
        ruta_nueva_lectura = Path(next(iter(archivos_nuevos_subidos)))
    else:
        ruta_nueva_lectura = Path(RUTA_BASE_NUEVA)
    if not ruta_nueva_lectura.is_file():
        raise FileNotFoundError(f"No existe {ruta_nueva_lectura}. Sube tu segundo archivo.")
    if FORMATO_BASE_NUEVA == "csv":
        df_nueva_original = pd.read_csv(ruta_nueva_lectura, sep=SEPARADOR_NUEVA, encoding=CODIFICACION_NUEVA, decimal=DECIMAL_NUEVA)
    elif FORMATO_BASE_NUEVA == "excel":
        df_nueva_original = pd.read_excel(ruta_nueva_lectura, sheet_name=HOJA_NUEVA, decimal=DECIMAL_NUEVA)
    else:
        raise ValueError("FORMATO_BASE_NUEVA debe ser csv o excel.")

if df_nueva_original.empty or not df_nueva_original.columns.is_unique:
    raise ValueError("La base nueva está vacía o tiene columnas repetidas.")
if any(str(c).startswith("ml_") for c in df_nueva_original.columns):
    raise ValueError("Renombra las columnas originales con prefijo ml_ para evitar sobrescribirlas.")
display(df_nueva_original.head())

### 15.1 Validar esquema y repetir las reglas declaradas
No se crean predictores faltantes. Para predecir no se exige el target; sus reglas se omiten si está ausente.
Las reglas de columnas ajenas al modelo se aplican cuando esas columnas estén presentes y se informa su ausencia.
Los nulos solo se aceptan si autorizaste la imputación correspondiente.
El cambio entre tipos numéricos compatibles (por ejemplo entero a decimal) se informa y se admite;
un predictor numérico leído como texto detiene la ejecución hasta que declares su conversión.

In [ ]:
columnas_requeridas_nuevas = [configuracion_cargada["fecha"]] if tipo_cargado == "serie_tiempo" else configuracion_cargada["predictores"]
columnas_faltantes_nuevas = sorted(set(columnas_requeridas_nuevas) - set(df_nueva_original.columns))
columnas_adicionales_nuevas = sorted(set(df_nueva_original.columns) - set(columnas_requeridas_nuevas) - {target_cargado} - set(configuracion_cargada["identificadores"]))
print("Predictores/fecha faltantes:", columnas_faltantes_nuevas)
print("Columnas adicionales que no se modelan:", columnas_adicionales_nuevas)
if columnas_faltantes_nuevas:
    raise ValueError("Faltan columnas requeridas. Corrige el archivo; no se crearán artificialmente.")

reglas_presentes_numericas = [c for c in configuracion_cargada["reglas_numericas"] if c in df_nueva_original]
reglas_presentes_textos = [c for c in configuracion_cargada["reglas_textos"] if c in df_nueva_original]
reglas_presentes_fechas = {c: f for c, f in configuracion_cargada["reglas_fechas"].items() if c in df_nueva_original}
reglas_presentes_reemplazos = {c: r for c, r in configuracion_cargada["reglas_reemplazos"].items() if c in df_nueva_original}
columnas_reglas_guardadas = set(configuracion_cargada["reglas_numericas"]) | set(configuracion_cargada["reglas_textos"]) | set(configuracion_cargada["reglas_fechas"]) | set(configuracion_cargada["reglas_reemplazos"])
print("Reglas omitidas por columnas no presentes:", sorted(columnas_reglas_guardadas - set(df_nueva_original.columns)))
df_nueva_reglas = aplicar_reglas(df_nueva_original, reglas_presentes_numericas, reglas_presentes_textos, reglas_presentes_fechas, reglas_presentes_reemplazos)

if tipo_cargado != "serie_tiempo":
    X_nueva = df_nueva_reglas.loc[:, configuracion_cargada["predictores"]].copy()
    comparacion_esquema_nuevo = pd.DataFrame({"entrenamiento": paquete_modelo_cargado["esquema"], "nuevo": X_nueva.dtypes.astype(str).to_dict()})
    display(comparacion_esquema_nuevo)
    for columna_nueva_numerica in configuracion_cargada["numericas"]:
        if not pd.api.types.is_numeric_dtype(X_nueva[columna_nueva_numerica]):
            raise TypeError(f"{columna_nueva_numerica} debe ser numérica. Revisa reglas y unidades.")
    for columna_nueva_categorica in configuracion_cargada["categoricas"]:
        tipos_categoria_nueva = {"texto" if isinstance(v, str) else "numero" if isinstance(v, (int, float, np.number, bool)) else "otro"
            for v in X_nueva[columna_nueva_categorica].dropna().unique()}
        tipo_categoria_guardado = paquete_modelo_cargado["esquema"][columna_nueva_categorica]
        categoria_guardada_numerica = pd.api.types.is_numeric_dtype(pd.api.types.pandas_dtype(tipo_categoria_guardado))
        if "otro" in tipos_categoria_nueva or len(tipos_categoria_nueva) > 1:
            raise TypeError(f"{columna_nueva_categorica} mezcla tipos incompatibles.")
        if tipos_categoria_nueva and categoria_guardada_numerica != (tipos_categoria_nueva == {"numero"}):
            raise TypeError(f"{columna_nueva_categorica} cambió entre códigos numéricos y texto.")
    if np.isinf(X_nueva[configuracion_cargada["numericas"]].to_numpy(dtype=float)).any():
        raise ValueError("La base nueva tiene infinitos.")
    if not configuracion_cargada["imputar_numericas"] and X_nueva[configuracion_cargada["numericas"]].isna().any().any():
        raise ValueError("Hay nulos numéricos y el modelo guardado no tiene imputación.")
    if not configuracion_cargada["imputar_categoricas"] and X_nueva[configuracion_cargada["categoricas"]].isna().any().any():
        raise ValueError("Hay nulos categóricos y el modelo guardado no tiene imputación.")
    if configuracion_cargada["categoricas"]:
        codificador_categorias_guardado = modelo_cargado.named_steps["preparacion"].named_transformers_["categoricas"].named_steps["codificacion"]
        categorias_desconocidas_nuevas = {c: [v for v in X_nueva[c].dropna().unique() if v not in conocidas]
            for c, conocidas in zip(configuracion_cargada["categoricas"], codificador_categorias_guardado.categories_)}
        print("Categorías desconocidas:", categorias_desconocidas_nuevas)
        if configuracion_cargada["categoria_desconocida"] == "error" and any(categorias_desconocidas_nuevas.values()):
            raise ValueError("Hay categorías desconocidas y configuraste detener la predicción.")
else:
    df_nueva_fechas = df_nueva_reglas.assign(**{configuracion_cargada["fecha"]:
        pd.to_datetime(df_nueva_reglas[configuracion_cargada["fecha"]], format=configuracion_cargada["formato_fecha"], errors="raise")})
    fechas_archivo_nuevo = pd.DatetimeIndex(df_nueva_fechas[configuracion_cargada["fecha"]])
    fechas_futuras_esperadas = pd.date_range(
        paquete_modelo_cargado["historia"].index[-1] + pd.tseries.frequencies.to_offset(configuracion_cargada["frecuencia"]),
        periods=len(df_nueva_fechas), freq=configuracion_cargada["frecuencia"])
    if not fechas_archivo_nuevo.equals(fechas_futuras_esperadas):
        raise ValueError("Las fechas nuevas deben ser consecutivas, ordenadas y comenzar después del origen guardado.")

### 15.2 Predecir sin volver a entrenar
Se llama únicamente a `predict`, `predict_proba` o a la función recursiva.
No se ejecuta `fit`, `fit_transform` ni una nueva codificación de categorías.
La salida conserva todos los campos originales y añade número de fila, predicción y probabilidades.

In [ ]:
if tipo_cargado == "clasificacion":
    codificador_target_cargado = paquete_modelo_cargado["codificador_target"]
    clases_cargadas = codificador_target_cargado.classes_
    probabilidades_nuevas = modelo_cargado.predict_proba(X_nueva)
    indice_positivo_cargado = int(codificador_target_cargado.transform([configuracion_cargada["clase_positiva"]])[0]) if len(clases_cargadas) == 2 else None
    codigos_prediccion_nueva = np.where(probabilidades_nuevas[:, indice_positivo_cargado] >= configuracion_cargada["umbral"],
        indice_positivo_cargado, 1 - indice_positivo_cargado) if len(clases_cargadas) == 2 else modelo_cargado.predict(X_nueva)
    etiquetas_prediccion_nueva = codificador_target_cargado.inverse_transform(codigos_prediccion_nueva.astype(int))
    nombres_probabilidades = {f"ml_prob_clase_{i}": probabilidades_nuevas[:, i] for i in range(len(clases_cargadas))}
    resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)),
        ml_prediccion=etiquetas_prediccion_nueva, **nombres_probabilidades)
    display(pd.DataFrame({"Columna": list(nombres_probabilidades), "Etiqueta": clases_cargadas}))
elif tipo_cargado == "regresion":
    predicciones_base_nueva = modelo_cargado.predict(X_nueva)
    resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)), ml_prediccion=predicciones_base_nueva)
else:
    predicciones_futuras_nuevas = pronostico_recursivo(modelo_cargado, paquete_modelo_cargado["historia"],
        fechas_futuras_esperadas, configuracion_cargada["rezagos"], configuracion_cargada["ventanas"])
    resultado_base_nueva = df_nueva_original.assign(ml_fila_origen=np.arange(len(df_nueva_original)), ml_prediccion=predicciones_futuras_nuevas.to_numpy())
    resultado_base_nueva.plot(x=configuracion_cargada["fecha"], y="ml_prediccion", figsize=(12, 4), title="Pronóstico futuro")
    plt.show()
display(resultado_base_nueva.head(20))

In [ ]:
if EVALUAR_BASE_NUEVA_CON_TARGET:
    if target_cargado not in df_nueva_reglas or df_nueva_reglas[target_cargado].isna().any():
        raise ValueError("Para evaluar necesitas target real no nulo en todas las filas nuevas.")
    if tipo_cargado == "clasificacion":
        clases_target_nuevo_desconocidas = set(df_nueva_reglas[target_cargado]) - set(clases_cargadas)
        if clases_target_nuevo_desconocidas:
            raise ValueError(f"El target nuevo contiene clases no entrenadas: {clases_target_nuevo_desconocidas}")
        reales_nuevos_codificados = codificador_target_cargado.transform(df_nueva_reglas[target_cargado])
        metricas_base_nueva = metricas_clasificacion(reales_nuevos_codificados, codigos_prediccion_nueva,
            probabilidades_nuevas, np.arange(len(clases_cargadas)), indice_positivo_cargado)
    else:
        if not pd.api.types.is_numeric_dtype(df_nueva_reglas[target_cargado]) or not np.isfinite(df_nueva_reglas[target_cargado].to_numpy(dtype=float)).all():
            raise ValueError("El target real nuevo debe ser numérico y finito.")
        metricas_base_nueva = metricas_regresion(df_nueva_reglas[target_cargado], resultado_base_nueva["ml_prediccion"])
    display(pd.Series(metricas_base_nueva, name="Base nueva").to_frame())

ruta_predicciones_exportadas = directorio_salida / "predicciones_base_nueva.csv"
resultado_base_nueva.to_csv(ruta_predicciones_exportadas, index=False, encoding="utf-8-sig")
print("Predicciones exportadas:", ruta_predicciones_exportadas)
# En Colab, opcionalmente:
# from google.colab import files
# files.download(str(ruta_predicciones_exportadas))
# files.download(str(ruta_modelo_guardado))

## 16. Qué revisar antes de usar el resultado
Escribe tus propias conclusiones: unidad de observación, target, limpieza aplicada, predictores disponibles
al predecir, modelo e hiperparámetros elegidos, métrica relevante y errores más costosos.
Compara siempre contra la línea base. Una brecha pequeña no demuestra por sí sola que un modelo generalice.
Revisa desempeño con datos posteriores y cambios en nulos, categorías o distribución de variables.

**Errores que esta guía deja visibles:** ruta incorrecta; target sin configurar; clases insuficientes;
tipos incompatibles; nulos sin política de imputación; fechas repetidas o huecos; horizonte sin historia;
predictores faltantes y categorías nuevas. Ninguno activa una limpieza alternativa silenciosa.

### Referencias utilizadas
- `S4_Taller_EDA_y_Modelado_vf.ipynb` y sus copias: calidad, distribuciones, relaciones, logística,
  árboles, Random Forest, búsquedas, SHAP, umbrales y scoring.
- `Copia de S3_Taller_EDA_ordenado_vf.ipynb`: secuencia del EDA.
- `Copia de S5_Prediccion_compra_RESUELTO.ipynb`: evaluación, deciles y predicción de compra.
- `Caso_RapiMenu_regresion.ipynb`: línea base, regresión y análisis de errores por segmento.
- `xgboost_lightGBM_mejoradovf.ipynb`: boosting, probabilidades y aplicación a observaciones nuevas.
- `Caso2_Series_de_Tiempo_Entidad_X.ipynb`: rezagos, pronóstico recursivo y backtesting.

Las decisiones específicas de churn, pedidos o feriados de esos casos no se trasladan automáticamente.
Consulta técnica: [Pipeline y ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html),
[XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html),
[SHAP Explainer](https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html).